# NLP Job Recommendation Engine

Semantic job recommendation system using sentence transformers and cosine similarity.  
Users submit a profile (skills, experience, preferences) and receive ranked job matches — served via a FastAPI endpoint.

**Stack:** Python · HuggingFace Sentence Transformers · FAISS · FastAPI · Pandas

---
## Pipeline
```
Job Descriptions → Sentence Transformer → FAISS Index
                                                ↑
User Profile → Sentence Transformer → Query Vector → Top-K Semantic Search → Ranked Results
```

## 1. Install Dependencies

In [ ]:
# !pip install sentence-transformers faiss-cpu fastapi uvicorn pandas numpy scikit-learn

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K      = 5
print("Imports OK")

## 3. Job Listings Dataset

In [ ]:
JOB_LISTINGS = [
    {"id": "JOB001", "title": "Senior Data Scientist", "company": "TechCorp", "location": "Bangalore",
     "description": "Build and deploy ML models for time-series forecasting and anomaly detection. Experience with Python, PySpark, Airflow, and cloud platforms (AWS, Azure). Knowledge of NLP and GenAI is a plus."},
    {"id": "JOB002", "title": "ML Engineer", "company": "DataAI", "location": "Hyderabad",
     "description": "Design and operationalise machine learning pipelines at scale. Strong knowledge of MLOps, Docker, Kubernetes, and CI/CD. Experience with PyTorch or TensorFlow required."},
    {"id": "JOB003", "title": "NLP Research Scientist", "company": "AI Labs", "location": "Remote",
     "description": "Research and develop NLP models including transformers, LLMs, and RAG systems. Publish findings and translate research into production systems using LangChain and vector databases."},
    {"id": "JOB004", "title": "Data Engineer", "company": "StreamBase", "location": "Pune",
     "description": "Build real-time data pipelines using Apache Spark, Kafka, and Databricks. Design data warehouse schemas in BigQuery and Snowflake. Python and SQL expertise required."},
    {"id": "JOB005", "title": "Analytics Engineer", "company": "InsightCo", "location": "Bangalore",
     "description": "Own analytics infrastructure and dashboards using Tableau and dbt. Work with stakeholders to define KPIs and build data models. SQL and Python proficiency required."},
    {"id": "JOB006", "title": "MLOps Engineer", "company": "CloudScale", "location": "Remote",
     "description": "Manage model lifecycle including training, deployment, and monitoring. Build CI/CD pipelines for ML using MLflow, Docker, and Kubernetes. AWS SageMaker and GCP Vertex AI experience preferred."},
    {"id": "JOB007", "title": "AI Product Data Scientist", "company": "AdTech Inc", "location": "Mumbai",
     "description": "Apply data science to ad measurement and audience analytics. Build statistical models for A/B testing and causal inference. Experience in ad tech or media measurement preferred."},
    {"id": "JOB008", "title": "Quantitative Analyst", "company": "FinTech", "location": "Bangalore",
     "description": "Develop quantitative models for risk and pricing using Bayesian statistics and regression. Python, R, and SQL required. Experience in financial modelling is a strong plus."},
    {"id": "JOB009", "title": "Computer Vision Engineer", "company": "VisionAI", "location": "Hyderabad",
     "description": "Build deep learning models for image classification and object detection using PyTorch. Deploy models on edge devices. Experience with OpenCV and ONNX preferred."},
    {"id": "JOB010", "title": "GenAI Engineer", "company": "LLMCo", "location": "Remote",
     "description": "Build production GenAI applications using LLMs, RAG pipelines, and prompt engineering. Experience with LangChain, vector databases (ChromaDB, Pinecone), and OpenAI API required."},
]

df_jobs = pd.DataFrame(JOB_LISTINGS)
df_jobs["full_text"] = df_jobs["title"] + ". " + df_jobs["description"]
print(f"Loaded {len(df_jobs)} job listings.")
df_jobs[["id", "title", "company", "location"]].head(10)

## 4. Load Embedding Model & Encode Jobs

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded: {MODEL_NAME}")

job_embeddings = model.encode(df_jobs["full_text"].tolist(), show_progress_bar=True)
print(f"Job embeddings shape: {job_embeddings.shape}")

## 5. FAISS Index for Fast Retrieval

In [ ]:
try:
    import faiss
    DIM = job_embeddings.shape[1]
    index = faiss.IndexFlatIP(DIM)   # Inner product (cosine after normalisation)
    # Normalise for cosine similarity
    norms = np.linalg.norm(job_embeddings, axis=1, keepdims=True)
    normed = job_embeddings / norms
    index.add(normed.astype(np.float32))
    USE_FAISS = True
    print(f"FAISS index built: {index.ntotal} vectors, dim={DIM}")
except ImportError:
    USE_FAISS = False
    print("FAISS not available — using sklearn cosine_similarity fallback.")

## 6. Recommendation Function

In [ ]:
def build_user_profile(skills, experience, preferences):
    """Convert user inputs to a single text representation."""
    return (
        f"Skills: {', '.join(skills)}. "
        f"Experience: {experience}. "
        f"Preferences: {preferences}."
    )

def recommend_jobs(profile_text, model, job_embeddings, df_jobs, top_k=5):
    """Encode profile and retrieve top-k semantically similar jobs."""
    query_vec = model.encode([profile_text])

    if USE_FAISS:
        q_norm = query_vec / np.linalg.norm(query_vec)
        scores, indices = index.search(q_norm.astype(np.float32), top_k)
        scores, indices = scores[0], indices[0]
    else:
        sims    = cosine_similarity(query_vec, job_embeddings)[0]
        indices = np.argsort(sims)[::-1][:top_k]
        scores  = sims[indices]

    results = df_jobs.iloc[indices].copy()
    results["similarity_score"] = scores.round(4)
    return results[["id", "title", "company", "location", "similarity_score"]].reset_index(drop=True)

print("Recommendation function ready.")

## 7. Example — Sachin's Profile

In [ ]:
profile_text = build_user_profile(
    skills=["Python", "Machine Learning", "NLP", "LLMs", "RAG", "LangChain",
            "Time-Series", "Anomaly Detection", "Bayesian Modeling",
            "Docker", "MLflow", "AWS", "Azure", "Databricks", "Airflow"],
    experience="4+ years as Senior Data Scientist at Nielsen, building production ML pipelines for audience measurement and ad analytics.",
    preferences="Senior Data Scientist or ML Engineer role. Preference for GenAI, NLP, or MLOps-heavy positions. Open to remote."
)

print("User Profile:")
print(profile_text)
print()

results = recommend_jobs(profile_text, model, job_embeddings, df_jobs, top_k=TOP_K)
print("Top Recommended Jobs:")
print(results.to_string(index=False))

## 8. Similarity Visualisation

In [ ]:
# Bar chart of top-k similarity scores
fig, ax = plt.subplots(figsize=(10, 4))
colors  = ["#1A5276" if s > 0.5 else "#AED6F1" for s in results["similarity_score"]]
ax.barh(results["title"] + " @ " + results["company"], results["similarity_score"], color=colors)
ax.set_xlabel("Cosine Similarity Score")
ax.set_title(f"Top {TOP_K} Job Recommendations")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("job_recommendations.png", dpi=150)
plt.show()

# Embedding similarity heatmap — all jobs vs profile
query_vec  = model.encode([profile_text])
all_scores = cosine_similarity(query_vec, job_embeddings)[0]

fig2, ax2 = plt.subplots(figsize=(12, 2))
im = ax2.imshow(all_scores.reshape(1, -1), aspect="auto", cmap="Blues")
ax2.set_xticks(range(len(df_jobs)))
ax2.set_xticklabels(df_jobs["title"], rotation=45, ha="right", fontsize=8)
ax2.set_yticks([])
ax2.set_title("Profile-to-Job Similarity Heatmap")
plt.colorbar(im, ax=ax2, fraction=0.02)
plt.tight_layout()
plt.savefig("similarity_heatmap.png", dpi=150)
plt.show()

## 9. FastAPI Endpoint Reference

See `serve.py` in this repo for the production-ready FastAPI server.

```bash
uvicorn serve:app --reload
# POST http://localhost:8000/recommend
```

### Example Request
```json
{
  "skills": ["Python", "NLP", "LLMs", "RAG", "Docker"],
  "experience": "4 years in ML and data science",
  "preferences": "Senior DS or GenAI role, remote preferred",
  "top_k": 5
}
```

### Example Response
```json
{
  "recommendations": [
    {"id": "JOB010", "title": "GenAI Engineer", "company": "LLMCo", "score": 0.89},
    {"id": "JOB003", "title": "NLP Research Scientist", "company": "AI Labs", "score": 0.85}
  ]
}
```